In [ ]:
import os
import gc
import sys

import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq

import pyro
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch.utils.data import random_split

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rcParams
from IPython.display import display

sns.set_context('paper')
rcParams.update({'font.family': 'Liberation Sans'})
rcParams.update({'font.size': 12})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [ ]:
sys.path.append('..')
sys.path.append('../models/')
sys.path.append('../util')

import IO, plot, utils, trajectory
import vgae, configs, dataset

%load_ext autoreload
%autoreload 2

#### Load data

In [ ]:
data_path = '../data/thymus/'
sample_id = 'Mouse_Thymus1'

adata_rna = sc.read_h5ad(os.path.join(data_path, sample_id, 'adata_rna.h5'))
adata_protein = sc.read_h5ad(os.path.join(data_path, sample_id, 'adata_protein.h5'))
adata_protein.var_names_make_unique()

In [ ]:
# Ground-truth CMA trajectory
fig, ax = plt.subplots()
ax = sq.pl.spatial_scatter(
    adata_rna, color='CMA',
    size=100, img=False, cmap='RdBu_r',
    title=None,
    ax=ax, return_ax=True
)
ax.set_title(r'Ground-truth spatial gradient $(t)$'+'\nalong Cortico-Medulla Axis (CMA)', fontsize=12)
plt.show()


#### LYNX

In [ ]:
n_latent = 6
adata_rna.obsm['X_z'] = np.load('../results/thymus/lynx_rna_{0}_{1}.npy'.format(n_latent, sample_id))
curve = trajectory.get_curve(adata_rna)
trajectory.compute_pseudotime(adata_rna, curve, root_marker='Dcn')

adata_protein.obsm['X_z'] = adata_rna.obsm['X_z'].copy()
adata_protein.obs['t'] = adata_rna.obs['t'].values

In [ ]:
# LYNX CMA trajectory
fig, ax = plt.subplots()
ax = sq.pl.spatial_scatter(
    adata_rna, color='t',
    size=100, img=False, cmap='RdBu_r',
    title=None,
    ax=ax, return_ax=True
)
ax.set_title(r'Inferred spatial gradient $(t)$ - LYNX', fontsize=12)
plt.show()

In [ ]:
t_lynx = adata_rna.obs['t'].values
t_true = adata_protein.obs['CMA'].values
t_true = (t_true-t_true.min()) / (t_true.max()-t_true.min())

plot.disp_kde_scatter(
    t_true, t_lynx, subset_ratio=0.2,
    logscale=False,
    xlabel=r"Ground-truth $(t)$",
    ylabel=r"LYNX prediction $(t)$",
    title="Spatial Gradient along CMA\n LYNX vs. Ground-truth"
)

#### SIMVI

In [ ]:
adata_rna.obsm['X_simvi'] = np.load('../results/thymus/SIMVI_rna_s20.npy')
curve = trajectory.get_curve(adata_rna, use_rep='X_simvi')
trajectory.compute_pseudotime(adata_rna, curve, root_marker='Dcn')

In [ ]:
adata_rna.obs['t'] = 1. - adata_rna.obs['t'].values  # Flip if oriented towards Cortex

In [ ]:
# SIMVI CMA trajectory
fig, ax = plt.subplots()
ax = sq.pl.spatial_scatter(
    adata_rna, color='t',
    size=100, img=False, cmap='RdBu_r',
    title=None,
    ax=ax, return_ax=True
)
ax.set_title(r'Inferred spatial gradient $(t)$ - SIMVI', fontsize=12)
plt.show()

In [ ]:
t_simvi = adata_rna.obs['t'].values
t_true = adata_protein.obs['CMA'].values
t_true = (t_true-t_true.min()) / (t_true.max()-t_true.min())

plot.disp_kde_scatter(
    t_true, t_simvi, subset_ratio=0.2,
    logscale=False,
    xlabel=r"Ground-truth $(t)$",
    ylabel=r"SIMVI prediction $(t)$",
    title="Spatial Gradient along CMA\n SIMVI vs. Ground-truth"
)

#### scVI

In [ ]:
adata_rna.obsm['X_scvi'] = np.load('../results/thymus/scvi_10.npy')

trajectory.get_curve(
    adata_rna, 
    use_rep='X_scvi',
    epg_mu=.01,
    root_marker='Dcn',
)

In [ ]:
# scVI CMA trajectory
fig, ax = plt.subplots()
ax = sq.pl.spatial_scatter(
    adata_rna, color='t',
    size=100, img=False, cmap='RdBu_r',
    title=None,
    ax=ax, return_ax=True
)
ax.set_title(r'Inferred spatial gradient $(t)$ - scVI', fontsize=12)
plt.show()

In [ ]:
t_scvi = adata_rna.obs['t'].values
t_true = adata_protein.obs['CMA'].values
t_true = (t_true-t_true.min()) / (t_true.max()-t_true.min())

plot.disp_kde_scatter(
    t_true, t_scvi, subset_ratio=0.2,
    logscale=False,
    xlabel=r"Ground-truth $(t)$",
    ylabel=r"scVI prediction $(t)$",
    title="Spatial Gradient along CMA\n scVI vs. Ground-truth"
)

#### TotalVI

In [ ]:
adata_rna.obsm['X_totalvi'] = np.load('../results/thymus/totalvi_20.npy')
curve = trajectory.get_curve(adata_rna, use_rep='X_totalvi')
trajectory.compute_pseudotime(adata_rna, curve, root_marker='Dcn')

In [ ]:
# totalVI CMA trajectory
fig, ax = plt.subplots()
ax = sq.pl.spatial_scatter(
    adata_rna, color='t',
    size=100, img=False, cmap='RdBu_r',
    title=None,
    ax=ax, return_ax=True
)
ax.set_title(r'Inferred spatial gradient $(t)$ - totalVI', fontsize=12)
plt.show()

In [ ]:
t_totalvi = adata_rna.obs['t'].values
t_true = adata_protein.obs['CMA'].values
t_true = (t_true-t_true.min()) / (t_true.max()-t_true.min())

plot.disp_kde_scatter(
    t_true, t_totalvi, subset_ratio=0.2,
    logscale=False,
    xlabel=r"Ground-truth $(t)$",
    ylabel=r"totalVI prediction $(t)$",
    title="Spatial Gradient along CMA\n totalVI vs. Ground-truth"
)